In [0]:
# Create choices list as a variable to avoid mismatch
schema_choices = ["bronze", "silver", "gold", "security"]

# Fix: Default values must be a string if multiple, or a list that matches choices
dbutils.widgets.text("project_catalog", "vstone_catalog", "1. Target Catalog Name")
dbutils.widgets.text("raw_schema", "raw", "2. Raw Schema Name")
dbutils.widgets.multiselect(
    "schemas", 
    "bronze", # Fixed: Providing a single valid default from the list
    schema_choices, 
    "3. Pipeline Schemas"
)

# Fetch values into variables
CATALOG = dbutils.widgets.get("project_catalog")
RAW_SCHEMA = dbutils.widgets.get("raw_schema")
# Get selected schemas and convert to list
SELECTED_SCHEMAS = dbutils.widgets.get("schemas").split(",")

In [0]:
# 1. Initialize Catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG} COMMENT 'VStone Russian Car Market Project'")

# 2. Dynamic Schema Creation
# Creating the raw landing schema first, then the pipeline layers
all_target_schemas = [RAW_SCHEMA] + SELECTED_SCHEMAS

for schema_name in all_target_schemas:
    spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name}
    COMMENT 'VStone {schema_name.upper()} layer'
    """)

# 3. Volume Creation (Landing and Chunks)
# Hardcoding avoided by using variables for schema and catalog
volumes_to_create = ["landing", "chunks"]

for vol in volumes_to_create:
    spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{RAW_SCHEMA}.{vol}
    COMMENT 'Raw zone: {vol} volume for car market pipeline'
    """)

print(f"""
✅ PROJECT RE-STARTED SUCCESSFULLY
--------------------------------------------------
Catalog Path: {CATALOG}
Raw Schema:   {RAW_SCHEMA}
Created Layers: {', '.join(SELECTED_SCHEMAS)}
Volumes:      /Volumes/{CATALOG}/{RAW_SCHEMA}/landing
              /Volumes/{CATALOG}/{RAW_SCHEMA}/chunks
--------------------------------------------------
""")

In [0]:
# # Notebook: src/notebooks/00_catalog_setup.py
# # Run ONCE before anything else
# CATALOG = "vstone_catalog"
# spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG} COMMENT 'VStone Russian Car Market Pipeline'")
# for schema in ["raw", "bronze", "silver", "gold", "security"]:
#     spark.sql(f"""
#     CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}
#     COMMENT 'VStone {schema.upper()} layer — Russian Car Market Project'
#     """)
# # Three volumes: landing (raw files), chunks (split outputs), checkpoints (streaming state)
# for volume in ["landing", "chunks"]:
#     spark.sql(f"""
#     CREATE VOLUME IF NOT EXISTS {CATALOG}.raw.{volume}
#     COMMENT 'Raw zone: {volume} volume for car market pipeline'
#     """)
# print(f"""
# Volumes created:
# /Volumes/{CATALOG}/raw/landing ← upload all 5 Kaggle files here
# /Volumes/{CATALOG}/raw/chunks ← chunked outputs go here
# """)